# CareSignal evidence, corrections and measured extraction

This notebook calls the same package functions as the Streamlit app. It replays the fictional two-resident correction flow, checks development inputs and inspects the saved 24-note extraction experiment without making API calls. The initial held-out run and the prompt-v2 post-hoc replay remain separate.

The references were translated and reviewed by coding agents, not human or clinical experts. The sample is small and deliberately stratified.

This project is a prototype and is not medical advice.

In [1]:
import sys
import tempfile
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "caresignal").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
from caresignal.evaluation import evaluate_examples, load_dataset, summary_markdown, write_results
from caresignal.extractors import FixtureExtractor, PhraseMatcher
from caresignal.fixtures import DEMO_REPLACEMENT
from caresignal.service import CareSignalService

## One note, two confirmed baselines

The note documents an aide beside the resident on the breakfast walk. Elena's confirmed usual level is Independent; Ruth's is Staff beside or supervising. The model does not compare the levels: code does.


In [3]:
demo_directory = tempfile.TemporaryDirectory(prefix="caresignal-notebook-")
service = CareSignalService(Path(demo_directory.name) / "demo.sqlite3", "notebook-fictional-demo")
service.reset_demo()
residents = {resident.display_name: resident for resident in service.list_residents()}
cases = {name: service.list_cases(resident.id)[0] for name, resident in residents.items()}
for case in cases.values():
    service.analyze_active_revision(case.id, FixtureExtractor())
views = {name: service.get_handoff_view(case.id) for name, case in cases.items()}
assert views["Elena"].revision.body == views["Ruth"].revision.body
assert views["Elena"].comparison.status == "MORE_HELP"
assert views["Ruth"].comparison.status == "MATCHES"
[
    (name, view.run.result.label.value, view.comparison.status, view.baseline.level.value)
    for name, view in views.items()
]

[('Elena', 'SUPERVISION', 'MORE_HELP', 'INDEPENDENT'),
 ('Ruth', 'SUPERVISION', 'MATCHES', 'SUPERVISION')]

## Replace the source, preserve the original

The replacement becomes active before inference starts. A walker alone leaves human assistance unspecified. The old flag and excerpt cannot remain on the current card.


In [4]:
original = views["Elena"].revision
replacement = service.replace_active_revision(cases["Elena"].id, original.id, DEMO_REPLACEMENT)
pending_view = service.get_handoff_view(cases["Elena"].id)
assert pending_view.revision.id == replacement.id
assert pending_view.run is None
service.analyze_active_revision(cases["Elena"].id, FixtureExtractor())
corrected_view = service.get_handoff_view(cases["Elena"].id)
assert corrected_view.comparison.status == "UNCLEAR"
assert corrected_view.run.revision_id == replacement.id
[
    (
        revision.version,
        "Active" if revision.id == corrected_view.revision.id else "Corrected",
        revision.body,
    )
    for revision in service.history(cases["Elena"].id)
]

[(2,
  'Active',
  'This morning she walked to breakfast with her walker. The note does not say whether anyone helped during the walk.'),
 (1,
  'Corrected',
  'This morning she needed an aide beside her on the way to breakfast. She ate her breakfast in the dining room.')]

## Frozen fictional development checks

These scores measure the phrase matcher's behavior on authored development examples, including deliberate distractors. They are not a held-out public experiment. Exact evidence requires the entire frozen span set, so a correct label with a different excerpt can fail the joint score.


In [5]:
examples = load_dataset(ROOT / "data/fictional_development.jsonl")
report = evaluate_examples(examples, PhraseMatcher())
print(summary_markdown(report))
artifacts = write_results(report, ROOT / "artifacts/notebook-fictional-development")
artifacts

| Mode | Model | Scope | N | Label | Exact evidence | Joint | Errors |
| --- | --- | --- | ---: | ---: | ---: | ---: | ---: |
| phrase | none | fictional_development | 7 | 7/7 (100.0%) | 5/7 (71.4%) | 5/7 (71.4%) | 0 |

Development results verify these examples only. They do not estimate held-out Fruta performance.



{'json': PosixPath('/workspace/work/caresignal/artifacts/notebook-fictional-development/results.json'),
 'csv': PosixPath('/workspace/work/caresignal/artifacts/notebook-fictional-development/results.csv'),
 'markdown': PosixPath('/workspace/work/caresignal/artifacts/notebook-fictional-development/summary.md')}

## Public synthetic development example: Fruta `nn_1093`

The original Dutch text and source dataset revision are frozen in `data/fruta_nn_1093.json`. The English text uses coding-agent-authored translation `caresignal-agent-en-literal-v1`, which has not been independently reviewed. This disclosed development example expects Hands-on help with evidence spanning both sentences. It is excluded from any held-out claim.


In [6]:
public_examples = load_dataset(ROOT / "data/fruta_development.jsonl")
public_report = evaluate_examples(public_examples, PhraseMatcher())
print(summary_markdown(public_report))
public_report["examples"][0]

| Mode | Model | Scope | N | Label | Exact evidence | Joint | Errors |
| --- | --- | --- | ---: | ---: | ---: | ---: | ---: |
| phrase | none | public_development | 1 | 1/1 (100.0%) | 1/1 (100.0%) | 1/1 (100.0%) | 0 |

Development results verify these examples only. They do not estimate held-out Fruta performance.



{'example_id': 'nn_1093',
 'split': 'development',
 'dataset_id': 'ekrombouts/fruta_records',
 'dataset_revision': '579141e3431c18a932fa3a0870e9f1c131f10367',
 'translation_id': 'caresignal-agent-en-literal-v1',
 'source_note_id': 'nn_1093',
 'source_url': 'https://huggingface.co/datasets/ekrombouts/fruta_records/blob/579141e3431c18a932fa3a0870e9f1c131f10367/data/train-00000-of-00001.parquet',
 'original_note': 'Mw. maakte een poging om zelf te lopen, maar stelde vast dat ze steun nodig had. We gaven haar fysieke ondersteuning en verduidelijkten de veiligheid.',
 'annotation_id': 'brief-development-expectation-v1',
 'note': 'The resident attempted to walk on her own, but found that she needed support. We gave her physical support and clarified safety.',
 'target': {'resident_id': 'fruta-cn_04',
  'resident_name': 'Fruta synthetic resident cn_04',
  'context_key': 'walking_attempt',
  'observed_on': '2024-03-07',
  'target_cue': 'attempt to walk on her own'},
 'gold_label': 'HANDS_ON',


## Recorded Gemini development check

The historical `gemini-3.8-flash` run on disclosed development example `nn_1093` returned Hands-on help with the exact two-sentence evidence. It is preserved in `docs/gemini-development-results/` and is excluded from held-out data. That model later failed to serve generation requests; the working default is now `gemini-3.1-flash-lite`.

The following cells inspect saved results. They do not contact a provider or need an API key.

## Frozen first-pass comparison

The dataset and agent review were committed before inference (`b28451c`). Both methods receive the same 24 translated notes and selected targets. The first Gemini run uses prompt `walking-assistance-v1`; all provider errors stay in the denominator. The earlier stopped provider attempt is archived separately.

Inspect the protocol in `docs/heldout-methods.md` and the complete trace in `docs/heldout-results/report.md`.

In [7]:
import json

from caresignal.evaluation import dataset_hash
from caresignal.reporting import write_evaluation_report

heldout_examples = load_dataset(ROOT / "data/fruta_held_out.jsonl")
first_pass = json.loads((ROOT / "docs/heldout-results/results.json").read_text())
assert len(heldout_examples) == 24
assert all(
    method["dataset"]["sha256"] == dataset_hash(heldout_examples)
    for method in first_pass["methods"]
)
print(summary_markdown(first_pass))
write_evaluation_report(
    first_pass,
    ROOT / "artifacts/notebook-first-pass-report.md",
    methods_note="Small stratified synthetic sample with coding-agent-reviewed references, not human clinical gold.",
)

| Mode | Model | Scope | N | Label | Exact evidence | Joint | Errors |
| --- | --- | --- | ---: | ---: | ---: | ---: | ---: |
| phrase | none | held_out_public | 24 | 14/24 (58.3%) | 12/24 (50.0%) | 8/24 (33.3%) | 0 |
| gemini | gemini-3.1-flash-lite | held_out_public | 24 | 11/24 (45.8%) | 14/24 (58.3%) | 7/24 (29.2%) | 0 |



PosixPath('/workspace/work/caresignal/artifacts/notebook-first-pass-report.md')

In [8]:
first_gemini = next(
    method for method in first_pass["methods"] if method["extractor"]["mode"] == "gemini"
)
example_error = next(row for row in first_gemini["examples"] if row["example_id"] == "na_0145")
assert not example_error["label_correct"]
{
    key: example_error[key]
    for key in (
        "source_note_id",
        "original_note",
        "note",
        "target",
        "gold_label",
        "predicted_label",
        "gold_evidence",
        "predicted_evidence",
    )
}

{'source_note_id': 'na_0145',
 'original_note': 'Dhr. is vanochtend geholpen met opstaan en aan kleden. Hij had wat moeite met lopen naar de gezamenlijke ruimte, maar met de rollator ging het redelijk. De tremoren zijn zichtbaar, vooral bij het vastpakken van de handgrepen.',
 'note': 'The resident was helped with getting up and dressing this morning. He had some difficulty walking to the shared room, but with the walker it went reasonably well. The tremors are visible, especially when grasping the handles.',
 'target': {'context_key': 'other_walk',
  'observed_on': '2024-06-05',
  'resident_id': 'fruta-ca_02',
  'resident_name': 'Fruta synthetic resident ca_02',
  'target_cue': 'walk to the shared room this morning'},
 'gold_label': 'UNCLEAR',
 'predicted_label': 'HELP_UNSPECIFIED',
 'gold_evidence': [{'end': 162,
   'quote': 'He had some difficulty walking to the shared room, but with the walker it went reasonably well.',
   'start': 67}],
 'predicted_evidence': [{'end': 162,
   'quo

## Prompt revision: post-hoc replay

The first-pass errors informed general fictional counterexamples for the shared `walking-assistance-v2` prompt. Reusing the same notes after that change is a **post-hoc development replay**, not a new held-out accuracy estimate. Frozen translations, labels and evidence remain unchanged.

Original rate-limited attempts and bounded retries are retained alongside the follow-up results. All 24 examples stay in the final denominator. See `docs/heldout-followup/` for the attempt history.

In [9]:
followup = json.loads((ROOT / "docs/heldout-followup/results.json").read_text())
assert all(method["scope"] == "post_hoc_replay" for method in followup["methods"])
assert all(
    method["dataset"]["sha256"] == dataset_hash(heldout_examples) for method in followup["methods"]
)
print(summary_markdown(followup))
next(method["summary"] for method in followup["methods"] if method["extractor"]["mode"] == "gemini")

| Mode | Model | Scope | N | Label | Exact evidence | Joint | Errors |
| --- | --- | --- | ---: | ---: | ---: | ---: | ---: |
| phrase | none | post_hoc_replay | 24 | 14/24 (58.3%) | 12/24 (50.0%) | 8/24 (33.3%) | 0 |
| gemini | gemini-3.1-flash-lite | post_hoc_replay | 24 | 21/24 (87.5%) | 16/24 (66.7%) | 16/24 (66.7%) | 0 |

Post-hoc replay on previously evaluated examples. These results are not a new held-out performance estimate.



{'completed': 24,
 'errors': 0,
 'evidence_accuracy': 0.6666666666666666,
 'evidence_correct': 16,
 'joint_accuracy': 0.6666666666666666,
 'joint_correct': 16,
 'label_accuracy': 0.875,
 'label_correct': 21,
 'total': 24}

## Reproduce a live replay explicitly

A live run requires `GEMINI_API_KEY` and may use provider quota. The notebook above uses saved actual predictions.

```bash
uv run --env-file .env python -m caresignal.evaluation --dataset data/fruta_held_out.jsonl --mode compare --provider gemini --post-hoc --min-request-interval 5 --output-dir artifacts/fruta-replay
```

Use a newly frozen, uninspected sample to measure any later prompt independently. The current frozen set remains useful for regression and error review.